# Module 2: Building an Honest Calendar

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

A group by only creates rows for things that happened. A month in which nothing
happened produces no row at all, so it silently vanishes from the series, and
every average computed afterwards is taken over the busy months only.

Worse, a month that vanished because nothing happened looks identical to a month
that vanished because the agency never submitted anything. The first is a zero.
The second is unknown. Averaging them together is wrong in two different
directions.

This notebook builds a calendar that keeps zeros, marks gaps, and sets aside the
months that are still being entered.

**About 15 minutes.**

## 1. The problem, in one agency

Orrindale Police Department has eight sworn officers and averages well under one
use of force incident a month.

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
from pathlib import Path

import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

print("reading from:", BASE)

uof = pd.read_csv(BASE + "use_of_force.csv")
elk = uof[uof["agency_id"] == "A006"]

seen = elk.groupby("year_month").size()
print(f"months that appear in the grouped result: {len(seen)}")

Fifty. But the dataset covers January 2019 through June 2026, which is ninety
months. Forty months are simply not there.

In [ ]:
calendar = pd.period_range("2019-01", "2026-06", freq="M").astype(str)
print(f"months in the calendar: {len(calendar)}")
print(f"months missing from the grouped result: {len(calendar) - len(seen)}")

Here is what those forty missing months cost. The mean over what the group by
returned is not the mean over the calendar, and the gap is large.

In [ ]:
full = seen.reindex(calendar, fill_value=0)

print(f"mean over the 50 months the group by returned: {seen.mean():.2f}")
print(f"mean over all 90 months in the calendar      : {full.mean():.2f}")
print(f"overstated by                                : "
      f"{100 * (seen.mean() / full.mean() - 1):.0f} percent")

## 2. Reindexing to a complete calendar

`reindex` is the fix. Give it every month you expect and it inserts the ones
that are absent.

The question is what to insert. For Orrindale, `fill_value=0` is right: the agency
reported every month and the honest answer was none. But that is a claim about
Orrindale, not a default, and the next agency shows why.

In [ ]:
print(full.head(14).to_string())

## 3. When zero is the wrong fill

Prairie County Sheriff's Office was migrating records systems in the spring of
2022 and never submitted three months. Those months are not zero. Nobody knows
what they were.

In [ ]:
pc = uof[uof["agency_id"] == "A009"]
pc_seen = pc.groupby("year_month").size()

absent = sorted(set(calendar) - set(pc_seen.index))
print(f"months absent from the event file: {len(absent)}")
print(absent)

Eleven months are absent, but only three of them are the migration. The other
eight are months in which Prairie County reported and had no use of force.

**The event file cannot tell you which is which.** An incident file only records
incidents; the absence of a record is not evidence of anything. You need a
second source that says which months the agency reported at all.

## 4. The reporting roster

In this dataset `agency_monthly.csv` serves that purpose. It has one row for
every agency and every month the agency submitted, whether or not anything
happened. Rows for the three migration months simply do not exist.

In [ ]:
am = pd.read_csv(BASE + "agency_monthly.csv")
pc_rows = am[am["agency_id"] == "A009"]

print(f"months Prairie County reported: {len(pc_rows)} of {len(calendar)}")
never_submitted = sorted(set(calendar) - set(pc_rows["year_month"]))
print(f"never submitted: {never_submitted}")

Now the eleven absent months split correctly: three are unknown, eight are
genuine zeros.

In [ ]:
genuine_zeros = sorted(set(absent) - set(never_submitted))
print(f"{len(genuine_zeros)} genuine zeros: {genuine_zeros}")

## 5. Building the series correctly

Reindex to the full calendar, fill the reported months with their counts,
including zero, and leave the unreported months as missing.

In [ ]:
reported = pd.Index(calendar).isin(pc_rows["year_month"])

series = (pc_seen.reindex(calendar)   # 90 months, NaN wherever the event file had none
                 .fillna(0)           # those NaNs become zeros
                 .where(reported))    # except the unreported ones, which go back to NaN

print(series.loc["2022-01":"2022-07"].to_string())
print(f"\nmissing values in the finished series: {int(series.isna().sum())}")

Compare the three ways this could have been handled:

In [ ]:
comparison = pd.DataFrame({
    "drop the absent months": [pc_seen.mean()],
    "fill everything with zero": [pc_seen.reindex(calendar, fill_value=0).mean()],
    "zeros as zero, gaps as missing": [series.mean()],
}, index=["mean incidents a month"]).round(3)
comparison.T

Filling the gaps with zero drags the mean down by inventing three quiet months.
Dropping the absent months pushes it up by discarding eight real zeros. Only the
third is right, and getting there required a source outside the incident file.

## 6. The last two months are not finished either

Records for recent months are still being entered. In this dataset the flag is
already provided; in a real extract you will usually have to ask.

In [ ]:
statewide = am.groupby("year_month").agg(
    calls=("total_cfs", "sum"), provisional=("provisional", "max"))

print(statewide.tail(5).to_string())

May and June 2026 are at roughly three quarters and one half of a normal month.
Nothing happened. Exclude them from anything you fit, and mark them on anything
you plot.

In [ ]:
final = am[am["provisional"] == 0]
print(f"rows before: {len(am):,}   after dropping provisional months: {len(final):,}")

## 7. A reusable function

Everything above, in one place. Use it at the top of any analysis in this
series.

In [ ]:
def monthly_series(events, roster, agency_id, start="2019-01", end="2026-06",
                   drop_provisional=True):
    """Counts per month for one agency: zeros kept, unreported months missing."""
    cal = pd.period_range(start, end, freq="M").astype(str)
    counts = events[events["agency_id"] == agency_id].groupby("year_month").size()

    rows = roster[roster["agency_id"] == agency_id]
    if drop_provisional:
        rows = rows[rows["provisional"] == 0]
    reported = pd.Index(cal).isin(rows["year_month"])

    return counts.reindex(cal).fillna(0).where(reported)

In [ ]:
s = monthly_series(uof, am, "A009")
print(f"length {len(s)}, missing {int(s.isna().sum())}, mean {s.mean():.3f}")
print(s.loc["2022-01":"2022-06"].to_string())

## 8. What to carry away

| Situation | What to do |
|---|---|
| The agency reported and nothing happened | keep it as `0` |
| The agency never reported | keep it as missing, never `0` |
| You cannot tell which | say so, and find the reporting roster |
| The month is still being entered | exclude it and label it provisional |

An average is only as honest as the calendar it is taken over.

## Exercise

Orrindale was treated as reporting every month. Verify that, then build its series
with the function above and compare the mean to the naive group by.

In [ ]:
# Fill in the blank, then run.
AGENCY = None              # try "A006"

if AGENCY:
    rows = am[am["agency_id"] == AGENCY]
    print(f"months reported: {len(rows)} of {len(calendar)}")
    s = monthly_series(uof, am, AGENCY)
    naive = uof[uof["agency_id"] == AGENCY].groupby("year_month").size()
    print(f"naive mean  {naive.mean():.3f}")
    print(f"honest mean {s.mean():.3f}")
else:
    print("Set AGENCY above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
AGENCY = "A006"
```

Orrindale reported all 90 months, so none of its absent months are gaps and every
one of them is a genuine zero. The naive mean is about 1.42 incidents a month
and the honest mean is about 0.81, because the naive version averages over busy
months only. Note that the honest mean also drops the two provisional months,
which is why it is not exactly the 0.79 you would get from the full 90.

A small agency is where this error does the most damage, since most of its
months are the ones that disappear.

</details>

---

**Next:** [Module 3, Choosing a Denominator](Module_03_Choosing_A_Denominator.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*